In [1]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification,Trainer,TrainingArguments
from datasets import load_dataset

In [2]:
dataset=load_dataset('json',data_files='train_pair_1w.json',split='train')
dataset

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['sentence1', 'sentence2', 'label'],
    num_rows: 10000
})

In [3]:
dataset['label'][0]

'1'

In [14]:
datasets = dataset.train_test_split(test_size=0.2)
datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label'],
        num_rows: 2000
    })
})

In [39]:
#数据处理
# 这次需要把数据集处理成：
# [cls] sentence1 [sep]
# [cls] sentence2 [sep]
# 计算两个 向量的 相似度 所以label 需要转换成1和-1

tokenizer = AutoTokenizer.from_pretrained('hfl/chinese-macbert-base')

def process_func(examples):
    sentences = []
    labels = []
    
    for sen1,sen2,label in zip(examples['sentence1'],examples['sentence2'],examples['label']):
        sentences.append(sen1)
        sentences.append(sen2)
        labels.append(1 if int(label)==1 else -1)
        
    tokenized_examples = tokenizer(sentences,max_length=128,truncation=True,padding='max_length')
    tokenized_examples = {k:[v[i:i+2] for i in range(0,len(v),2)] for k,v in tokenized_examples.items()}
    tokenized_examples['labels'] = labels
    
    return tokenized_examples


tokenized_datasets = datasets.map(process_func,batched=True,remove_columns=datasets['train'].column_names)
tokenized_datasets

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})

In [40]:
print(tokenized_datasets['train']['labels'])

[1, -1, -1, -1, -1, -1, 1, -1, 1, -1, 1, 1, -1, -1, 1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 1, -1, -1, -1, -1, -1, -1, -1, -1, -1, 1, -1, -1, -1, 1, -1, -1, 1, 1, -1, 1, 1, -1, -1, -1, -1, 1, -1, -1, -1, -1, 1, 1, -1, -1, -1, 1, 1, -1, -1, 1, 1, -1, 1, -1, -1, 1, -1, 1, -1, -1, -1, -1, 1, 1, 1, -1, -1, -1, -1, -1, -1, 1, -1, -1, 1, -1, -1, -1, -1, 1, 1, 1, -1, 1, -1, -1, -1, -1, -1, 1, 1, -1, -1, -1, -1, -1, 1, 1, -1, 1, -1, -1, 1, 1, 1, -1, 1, 1, -1, -1, 1, 1, 1, -1, 1, 1, 1, 1, 1, 1, 1, 1, -1, 1, -1, -1, 1, 1, -1, -1, -1, 1, -1, 1, -1, -1, -1, 1, -1, -1, 1, 1, 1, -1, 1, 1, -1, -1, -1, 1, -1, -1, 1, -1, -1, 1, -1, -1, 1, -1, -1, -1, 1, 1, 1, -1, 1, -1, -1, -1, -1, -1, 1, 1, 1, 1, -1, -1, -1, -1, 1, -1, -1, 1, 1, -1, -1, -1, -1, -1, -1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1, 1, 1, -1, 1, -1, 1, -1, -1, -1, -1, -1, -1, 1, -1, -1, 1, -1, -1, 1, 1, 1, 1, -1, -1, -1, -1, -1, -1, 1, 1, -1, 1, 1, 1, -1, -1, -1, 1, 1, -1, -1, 1, 1, -1, 1, 1, -1, -1, 1, 1, 1, -1, -1, 1, -1, -1, -1, -1, 1, 1, 

In [41]:
from transformers import BertPreTrainedModel,BertModel,BertForSequenceClassification
import torch
from typing import Optional
from torch.nn import CosineSimilarity, CosineEmbeddingLoss

class DualModel(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config
        self.bert = BertModel(config)
        self.post_init()
    
    def forward(
        self,
        input_ids: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        token_type_ids: Optional[torch.Tensor] = None,
        position_ids: Optional[torch.Tensor] = None,
        head_mask: Optional[torch.Tensor] = None,
        inputs_embeds: Optional[torch.Tensor] = None,
        labels: Optional[torch.Tensor] = None,
        output_attentions: Optional[bool] = None,
        output_hidden_states: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        
        
        # 输入有 input_ids, attention_mask, token_type_ids
        #获得sen1 和 sen2 的输入
        senA_input_ids, senB_input_ids = input_ids[:,0],input_ids[:,1]
        senA_attention_mask, senB_attention_mask = attention_mask[:,0],attention_mask[:,1]
        senA_token_type_ids, senB_token_type_ids = token_type_ids[:,0],token_type_ids[:,1]
        
        # 获得向量表示
        sen_A_outputs = self.bert(
            senA_input_ids,
            attention_mask=senA_attention_mask,
            token_type_ids=senA_token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        
        sen_A_pooled = sen_A_outputs[1]
        
        sen_B_outputs = self.bert(
            senB_input_ids,
            attention_mask=senB_attention_mask,
            token_type_ids=senB_token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        
        sen_B_pooled = sen_B_outputs[1]
        
        cos = CosineSimilarity()(sen_A_pooled,sen_B_pooled)
        
        loss = None
        labels = labels.view(-1)
        if labels is not None:
            loss_fct = CosineEmbeddingLoss(0.3)
            loss = loss_fct(sen_A_pooled,sen_B_pooled,labels)
        
        output = (cos,)
        return ((loss,) + output) if loss is not None else output
        # 推理时只输出 cos 训练时 输出 loss + cos
        
model = DualModel.from_pretrained('hfl/chinese-macbert-base')

In [42]:
import evaluate

acc_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

In [43]:
def eval_metric(eval_predict):
    predictions, labels = eval_predict
    predictions = [int(p>0.7) for p in predictions]
    labels = [int(l>0) for l in labels]
    acc = acc_metric.compute(predictions = predictions,references = labels)
    f1 = f1_metric.compute(predictions =predictions,references = labels)
    acc.update(f1)
    
    return acc

In [44]:
train_args = TrainingArguments(output_dir="./dual_model",      # 输出文件夹
                               per_device_train_batch_size=32,  # 训练时的batch_size
                               per_device_eval_batch_size=32,   # 验证时的batch_size
                               logging_steps=10,                # log 打印的频率
                               eval_strategy="epoch",           # 评估策略
                               save_strategy="epoch",           # 保存策略
                               save_total_limit=3,              # 最大保存数
                               learning_rate=2e-5,              # 学习率
                               weight_decay=0.01,               # weight_decay
                               metric_for_best_model="f1",      # 设定评估指标
                               load_best_model_at_end=True)     # 训练完成后加载最优模型

In [45]:
trainer = Trainer(model=model, 
                  args=train_args, 
                  tokenizer=tokenizer,
                  train_dataset=tokenized_datasets["train"].select(range(100)), 
                  eval_dataset=tokenized_datasets["test"].select(range(100)), 
                  compute_metrics=eval_metric)

/var/folders/zv/x4vdjf_9115_6x3p0ybt2qzjzgmldp/T/ipykernel_14495/305272890.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model,


In [46]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.387630,0.380000,0.544118
2,No log,0.366089,0.430000,0.558140


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.387630,0.380000,0.544118
2,No log,0.366089,0.430000,0.558140
3,0.399500,0.353622,0.460000,0.564516


TrainOutput(global_step=12, training_loss=0.377318079272906, metrics={'train_runtime': 1421.3242, 'train_samples_per_second': 0.211, 'train_steps_per_second': 0.008, 'total_flos': 39465949593600.0, 'train_loss': 0.377318079272906, 'epoch': 3.0})

In [47]:
trainer.evaluate(tokenized_datasets["test"])

{'eval_loss': 0.42174991965293884,
 'eval_accuracy': 0.3975,
 'eval_f1': 0.5688729874776386,
 'eval_runtime': 58.8125,
 'eval_samples_per_second': 34.006,
 'eval_steps_per_second': 1.071,
 'epoch': 3.0}

In [63]:
class SentenceSimilarityPipeline:

    def __init__(self, model, tokenizer) -> None:
        self.model = model.bert
        self.tokenizer = tokenizer
        self.device = model.device

    def preprocess(self, senA, senB):
        return self.tokenizer([senA, senB], max_length=128, truncation=True, return_tensors="pt", padding=True)

    def predict(self, inputs):
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        return self.model(**inputs)[1]  # [2, 768]

    def postprocess(self, logits):
        cos = CosineSimilarity()(logits[None, 0, :], logits[None,1, :]).squeeze().cpu().item()
        return cos

    def __call__(self, senA, senB, return_vector=False):
        inputs = self.preprocess(senA, senB)
        logits = self.predict(inputs)
        result = self.postprocess(logits)
        if return_vector:
            return result, logits
        else:
            return result

In [64]:
pipe = SentenceSimilarityPipeline(model,tokenizer)

In [68]:
pipe("我喜欢北京", "你喜欢吃什么", return_vector=True)

(1.0,
 tensor([[ 0.0399,  0.0252,  0.0461,  ..., -0.0463, -0.0505, -0.0176],
         [ 0.0399,  0.0252,  0.0461,  ..., -0.0463, -0.0505, -0.0176]],
        device='mps:0', grad_fn=<TanhBackward0>))